# Tutorial Activo: Transformers y Atención con PyTorch

Este notebook está diseñado para **aprender construyendo**.

No se trata de ejecutar celdas de principio a fin: en varios puntos tendrás que:

1. Completar funciones incompletas (`TODO`).
2. Corregir bugs intencionales.
3. Modificar decisiones de diseño y comparar resultados.
4. Justificar tus decisiones con evidencia (métricas, gráficas o logs).

## Ruta de aprendizaje (basada en Stages 10-16)

- Stage 10: De teoría a implementación (single-head y multi-head).
- Stage 11: Batching, padding y máscaras causales.
- Stage 12: Design choices (pre/post norm, activaciones, codificación posicional).
- Stage 13: Dinámica de entrenamiento (gradientes, warmup, early stopping).
- Stage 14: Debugging e interpretación de atención.
- Stage 15-16: Comportamientos emergentes y mini-proyecto de análisis.

## Reglas de trabajo para estudiantes

- Si una celda tiene `TODO`, no la saltes.
- Si una prueba falla, **no** edites la prueba: corrige tu implementación.
- En cada sección de retos, escribe una breve conclusión en markdown:
  - Qué cambiaste.
  - Qué observaste.
  - Por qué crees que ocurrió.

## Convención

- `batch_size = B`
- `seq_len = T`
- `d_model = D`
- `num_heads = H`
- `d_k = D/H`

In [ ]:
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)

In [ ]:
def check_close(name, got, expected, atol=1e-6, rtol=1e-5):
    if isinstance(got, torch.Tensor):
        ok = torch.allclose(got, expected, atol=atol, rtol=rtol)
    else:
        ok = np.allclose(got, expected, atol=atol, rtol=rtol)
    print(f"{name}: {'OK' if ok else 'FAIL'}")
    if not ok:
        raise AssertionError(f"{name} no coincide con el valor esperado")


def check_shape(name, tensor, expected_shape):
    ok = tuple(tensor.shape) == tuple(expected_shape)
    print(f"{name} shape {tuple(tensor.shape)}: {'OK' if ok else 'FAIL'}")
    if not ok:
        raise AssertionError(f"{name}: shape esperada {expected_shape}, recibida {tuple(tensor.shape)}")

## Stage 10a: Single-Head Attention en NumPy

### Objetivo
Implementar atención escalada desde cero para entender exactamente:

1. `scores = QK^T / sqrt(d_k)`
2. `weights = softmax(scores)`
3. `output = weights V`

### Ejercicio 10a.1
Completa las funciones con `TODO` y valida con las pruebas de la siguiente celda.

In [ ]:
def softmax_np(x, axis=-1):
    """TODO: Implementa softmax numéricamente estable."""
    # 1) Resta el máximo por eje para estabilidad
    # 2) Aplica exp
    # 3) Divide entre la suma por eje
    raise NotImplementedError("Implementa softmax_np")


def scaled_dot_product_attention_np(Q, K, V, mask=None):
    """
    Q: (T, d_k)
    K: (T, d_k)
    V: (T, d_v)
    mask: (T, T) con True en posiciones permitidas
    """
    # TODO 1: calcula d_k
    # TODO 2: scores = Q @ K.T / sqrt(d_k)
    # TODO 3: aplica mask usando -inf en posiciones no permitidas
    # TODO 4: weights = softmax_np(scores, axis=-1)
    # TODO 5: output = weights @ V
    raise NotImplementedError("Implementa scaled_dot_product_attention_np")

In [ ]:
# Pruebas para Ejercicio 10a.1
T, d_k, d_v = 4, 2, 2
x = np.array([
    [0.1, 0.5],
    [0.8, 0.1],
    [0.4, 0.6],
    [0.2, 0.9],
])

rng = np.random.default_rng(123)
W_Q = rng.normal(size=(d_k, d_k)) * 0.1
W_K = rng.normal(size=(d_k, d_k)) * 0.1
W_V = rng.normal(size=(d_k, d_v)) * 0.1

Q = x @ W_Q.T
K = x @ W_K.T
V = x @ W_V.T

causal_mask = np.tril(np.ones((T, T))).astype(bool)

# Test de softmax
raw = np.array([[1.0, 2.0, 3.0]])
expected_softmax = np.exp(raw - np.max(raw)) / np.sum(np.exp(raw - np.max(raw)))

try:
    got_softmax = softmax_np(raw, axis=-1)
    check_close("softmax_np", got_softmax, expected_softmax)
except NotImplementedError as e:
    print(e)

# Test de atención
try:
    out, w = scaled_dot_product_attention_np(Q, K, V, mask=causal_mask)
    print("Shape output:", out.shape)
    print("Shape weights:", w.shape)
    assert out.shape == (T, d_v)
    assert w.shape == (T, T)
    check_close("Suma por fila en weights", w.sum(axis=-1), np.ones(T), atol=1e-5)
    print("Fila 0 (debe atender solo a sí mismo):", w[0])
except NotImplementedError as e:
    print(e)

### Ejercicio 10a.2 (Debugging)

En modelos reales, olvidar el factor `1/sqrt(d_k)` hace que el softmax se vuelva demasiado picudo.

1. Implementa la función de abajo.
2. Compara la entropía de la distribución con y sin escalamiento.
3. Explica en 3-5 líneas por qué eso afecta gradientes.

In [ ]:
def entropy_np(p, eps=1e-12):
    p = np.clip(p, eps, 1.0)
    return -(p * np.log(p)).sum(axis=-1).mean()


def compare_scaled_vs_unscaled(Q, K):
    """TODO: devuelve (entropy_unscaled, entropy_scaled)."""
    raise NotImplementedError("Implementa compare_scaled_vs_unscaled")


try:
    e_unscaled, e_scaled = compare_scaled_vs_unscaled(Q, K)
    print(f"Entropia sin escalamiento: {e_unscaled:.4f}")
    print(f"Entropia con escalamiento: {e_scaled:.4f}")
except NotImplementedError as e:
    print(e)

## Stage 10b: Multi-Head Attention en PyTorch

### Objetivo
Implementar MHA cuidando la parte más propensa a bugs:

1. Proyecciones lineales Q/K/V.
2. `view + transpose` para separar cabezas.
3. Aplicación correcta de máscara.
4. Concatenación de cabezas y proyección final.

### Ejercicio 10b.1
Completa la clase siguiente y pasa las pruebas.

In [ ]:
class MultiHeadAttentionStudent(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model debe ser divisible por num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        # x: (B, T, D)
        B, T, _ = x.shape
        # TODO: convierte a (B, H, T, d_k)
        raise NotImplementedError("Implementa split_heads")

    def combine_heads(self, x):
        # x: (B, H, T, d_k)
        B, H, T, d_k = x.shape
        # TODO: convierte a (B, T, D)
        raise NotImplementedError("Implementa combine_heads")

    def forward(self, Q, K, V, mask=None):
        # Q, K, V: (B, T, D)
        # mask: (B, 1, T, T) o (B, 1, 1, T)

        # TODO 1: proyecciones lineales
        # TODO 2: separar cabezas
        # TODO 3: scores = Q @ K^T / sqrt(d_k)
        # TODO 4: aplicar mask con masked_fill
        # TODO 5: attention = softmax(scores)
        # TODO 6: salida por cabeza y luego concatenar
        # TODO 7: proyeccion final W_O
        raise NotImplementedError("Implementa forward")

In [ ]:
# Pruebas para Ejercicio 10b.1
B, T, D, H = 2, 5, 16, 4
mha = MultiHeadAttentionStudent(D, H).to(device)

x = torch.randn(B, T, D, device=device)
mask_causal = torch.tril(torch.ones(T, T, device=device)).unsqueeze(0).unsqueeze(0)

try:
    y, w = mha(x, x, x, mask=mask_causal)
    check_shape("MHA output", y, (B, T, D))
    check_shape("MHA weights", w, (B, H, T, T))

    row_sums = w.sum(dim=-1)
    check_close("Suma por fila en attention weights", row_sums, torch.ones_like(row_sums), atol=1e-5)
except NotImplementedError as e:
    print(e)

## Stage 11: Batching, Padding y Máscaras

En teoría usamos una secuencia. En práctica usamos lotes con longitudes variables.

### Ejercicio 11.1
Implementa:

1. Máscara de padding: evita atender a `<PAD>`.
2. Máscara causal: evita atender al futuro.
3. Combinación de ambas máscaras.

In [ ]:
def create_padding_mask(sequences, pad_token_id=0):
    """
    sequences: (B, T)
    return: (B, 1, 1, T) con 1 en tokens reales, 0 en padding
    """
    # TODO
    raise NotImplementedError("Implementa create_padding_mask")


def create_causal_mask(seq_len, device):
    """
    return: (1, 1, T, T)
    """
    # TODO
    raise NotImplementedError("Implementa create_causal_mask")


def create_full_mask(sequences, pad_token_id=0):
    """
    Combina padding y causal.
    return: (B, 1, T, T)
    """
    # TODO
    raise NotImplementedError("Implementa create_full_mask")

In [ ]:
# Pruebas para Ejercicio 11.1
seq = torch.tensor([
    [5, 9, 2, 0, 0],
    [4, 7, 8, 1, 0],
], device=device)

try:
    pm = create_padding_mask(seq, pad_token_id=0)
    cm = create_causal_mask(seq_len=seq.shape[1], device=device)
    fm = create_full_mask(seq, pad_token_id=0)

    check_shape("padding_mask", pm, (2, 1, 1, 5))
    check_shape("causal_mask", cm, (1, 1, 5, 5))
    check_shape("full_mask", fm, (2, 1, 5, 5))

    # Token en padding no debe ser atendible en eje de keys
    print("full_mask[0,0,4]:", fm[0, 0, 4])
except NotImplementedError as e:
    print(e)

### Ejercicio 11.2: Cálculo de eficiencia

Completa la función para estimar FLOPs de `Q @ K^T` y responde:

1. ¿Cómo cambia el costo si duplicas `seq_len`?
2. ¿Cuál es el principal cuello de botella de memoria?

In [ ]:
def attention_flops(batch_size, seq_len, d_k, num_heads):
    """TODO: regresa (flops_por_cabeza, flops_totales)."""
    raise NotImplementedError("Implementa attention_flops")


try:
    fh, ft = attention_flops(batch_size=32, seq_len=512, d_k=64, num_heads=8)
    print("FLOPs por cabeza:", fh)
    print("FLOPs totales:", ft)
except NotImplementedError as e:
    print(e)

## Stage 12: Design Choices y sus implicaciones

En esta parte vas a experimentar variaciones arquitectónicas y observar impacto en estabilidad y convergencia.

### Ejercicio 12.1
Implementa un bloque Transformer configurable con:

- `norm_type`: `pre` o `post`
- `activation`: `relu` o `gelu`

Luego compara pérdidas en un mini-problema sintético.

In [ ]:
class TinyTransformerBlock(nn.Module):
    def __init__(self, d_model=32, num_heads=4, d_ff=64, norm_type="pre", activation="gelu"):
        super().__init__()
        self.norm_type = norm_type
        self.activation = activation

        self.attn = MultiHeadAttentionStudent(d_model=d_model, num_heads=num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff1 = nn.Linear(d_model, d_ff)
        self.ff2 = nn.Linear(d_ff, d_model)

    def ff(self, x):
        if self.activation == "relu":
            return self.ff2(F.relu(self.ff1(x)))
        elif self.activation == "gelu":
            return self.ff2(F.gelu(self.ff1(x)))
        else:
            raise ValueError("activation debe ser 'relu' o 'gelu'")

    def forward(self, x, mask=None):
        if self.norm_type == "post":
            # TODO: implementar versión post-norm
            raise NotImplementedError("Implementa forward para post-norm")
        elif self.norm_type == "pre":
            # TODO: implementar versión pre-norm
            raise NotImplementedError("Implementa forward para pre-norm")
        else:
            raise ValueError("norm_type debe ser 'pre' o 'post'")

## Stage 13: Dinámica de entrenamiento

### Ejercicio 13.1
Implementa un scheduler tipo warmup + decay y úsalo en un ciclo de entrenamiento corto.

Recordatorio:

- warmup: crecimiento lineal
- decay: proporcional a `step^-0.5`

In [ ]:
class WarmupSchedulerStudent:
    def __init__(self, d_model, warmup_steps):
        self.d_model = d_model
        self.warmup_steps = warmup_steps

    def get_lr(self, step):
        # TODO: implementar formula de Vaswani
        raise NotImplementedError("Implementa get_lr")


def log_gradient_stats(model):
    stats = {}
    for name, p in model.named_parameters():
        if p.grad is not None:
            stats[name] = p.grad.data.norm(2).item()
    return stats


try:
    sched = WarmupSchedulerStudent(d_model=512, warmup_steps=4000)
    for s in [1, 100, 1000, 4000, 8000]:
        print(s, sched.get_lr(s))
except NotImplementedError as e:
    print(e)

## Stage 14: Debugging e intuición

### Ejercicio 14.1 (Shape tracking)

Agrega prints o asserts para verificar que en todo bloque Transformer la shape permanezca `(B, T, D)`.

### Ejercicio 14.2 (Visualización de atención)

Grafica un mapa de calor por cabeza y describe:

1. Si hay diagonal dominante.
2. Si hay patrones fuera de diagonal.
3. Qué hipótesis lingüística/estructural propones para esa cabeza.

In [ ]:
def visualize_attention(weights, tokens=None, head=0, batch_idx=0):
    """
    weights: (B, H, T, T)
    """
    w = weights[batch_idx, head].detach().cpu().numpy()

    plt.figure(figsize=(6, 5))
    plt.imshow(w, cmap="Blues", aspect="auto")
    plt.colorbar(label="attention")
    plt.title(f"Head {head}")
    plt.xlabel("Key positions")
    plt.ylabel("Query positions")

    if tokens is not None and len(tokens) == w.shape[0]:
        plt.xticks(range(len(tokens)), tokens, rotation=45)
        plt.yticks(range(len(tokens)), tokens)

    plt.tight_layout()
    plt.show()


print("Usa esta función después de implementar MultiHeadAttentionStudent y ejecutar un forward.")

## Stage 15-16: Mini-proyecto y comportamientos emergentes

### Reto integrador (en equipos)

Construye un mini modelo de lenguaje autoregresivo (toy) con tu bloque Transformer y responde:

1. ¿Qué cabeza parece capturar dependencia local?
2. ¿Qué cabeza parece más global?
3. ¿Qué pasa al remover una cabeza específica (ablación)?

### Entregables

1. Implementaciones completas de todos los `TODO`.
2. Capturas de pruebas pasando.
3. 2 gráficas de atención con interpretación.
4. Comparativa breve: pre vs post norm, ReLU vs GELU.
5. Nota técnica de debugging: el bug más difícil y cómo lo resolvieron.

### Rubrica sugerida

- Correctitud de implementación: 40%
- Capacidad de debugging: 20%
- Calidad de análisis de atención: 20%
- Justificación de decisiones de diseño: 20%

## Planeación sugerida por semanas

| Semana | Contenido | Producto |
|---|---|---|
| 1 | Atención single-head (NumPy) + estabilidad numérica | Implementación y pruebas de Stage 10a |
| 2 | Multi-head attention en PyTorch | Implementación y pruebas de Stage 10b |
| 3 | Máscaras y batching | Demostración de Stage 11 con casos límite |
| 4 | Design choices | Comparativa pre/post norm y ReLU/GELU |
| 5 | Entrenamiento y monitoreo | Scheduler + estadísticas de gradiente |
| 6 | Debugging e interpretación | Mini-proyecto, visualizaciones y reporte |

## Ideas clave para reforzar en clase

- Atención elimina el cuello de botella de compresión secuencial de RNN.
- Q/K/V permiten consultas flexibles sobre el contexto.
- Multi-head favorece especialización funcional.
- Residuales habilitan profundidad efectiva.
- El costo dominante de atención es cuadrático en longitud de secuencia.